# **CSCI 161.03 - Introduction to Social Computing**

**Author:** BINWAG, Louis III G. (`github/louis-uwie`)

# **I. Introduction**

As part of the major requirement for **CSCI 161.03 - Introduction to Social Computing**, the author wants to dabble into social dynamics in subreddit discussions that concern the Philippines.

In `r/truePhilippines`, users tend to interact about on-going controversial issues in the Philippines. This ranges from the most recent `Nepo Babies` to rather old but still very relevant and recurring, `Agricultural Prices`.

**What I want to find out are:**

1. What are the **recurring themes** and topics in these discussions?
2. What **dominant emotions** or sentiments emerge in online conversations about the Philippines?
3. **[NOT YET FINAL, STILL THINKING ON SUBJECT SCOPE]** How do users interact with one another *(e.g., patterns of agreement, disagreement, or polarization)* when engaging in these topics?

# **II. About the Dataset**

 Please note that there is a great chance of including other social media platforms as well due to rate-limits of Reddit API Scraping.

Currently, there is `248 Rows`, and `8 Columns`.


| Column           | Definition                                      | Sample                                                              |
| ---------------- | ----------------------------------------------- | ------------------------------------------------------------------- |
| `post_title`     | Title of the Reddit post                        | *"Why is Python so popular in data science?"*                       |
| `post_url`       | Direct link to the Reddit post                  | `https://www.reddit.com/r/datascience/comments/abc123/...`          |
| `post_body`      | Main body text of the Reddit post (if provided) | *"I’ve noticed a lot of job postings require Python experience..."* |
| `post_author`    | Username of the Reddit post’s author            | `u/data_guru`                                                       |
| `post_score`     | Upvotes minus downvotes for the post            | `1542`                                                              |
| `comment`        | Text of the Reddit comment                      | *"Python has great libraries like pandas and scikit-learn."*        |
| `comment_author` | Username of the Reddit comment’s author         | `u/ml_enthusiast`                                                   |
| `comment_score`  | Upvotes minus downvotes for the comment         | `237`                                                               |



# **III. Data Collection & Preprocessing**

Data collection will primarily utilize Reddit's PRAW (API) for datascraping. Connecting to Google drive for ease of use since the author will be using google collab for a more streamline and 'smoother' project.


In [1]:
!pip install praw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 5.0 MB/s eta 0:00:00


In [2]:
# from dotenv import load_dotenv
# import os

# load_dotenv("/content/drive/CSCI 161/Project/redditdatascrape.env", override=True)

# CLIENT_ID = os.getenv("CLIENT_ID")
# CLIENT_SECRET = os.getenv("CLIENT_SECRET")

# print("CLIENT_ID:", CLIENT_ID)
# print("CLIENT_SECRET:", CLIENT_SECRET)


In [3]:
import pandas as pd
import numpy as np
import nltk

import praw

import pandas as pd
from google.colab import drive
import datetime

import os

In [4]:
reddit = praw.Reddit(
    client_id="390Mv4deA5sjduaY4-n2TA",
    client_secret="-lvWJJC94wvX8-7DLNrp0wgaa70EOA",
    user_agent="script:reddit_scraper:v1.0"
)

print("Read-only mode:", reddit.read_only)


Read-only mode: True


In [5]:
# Mount Google Drive
drive.mount('/content/drive')

subreddit = reddit.subreddit("truePhilippines")

posts_data = []

# Fetch more posts (adjust as needed)
for post in subreddit.hot(limit=200):
    body = post.selftext if post.selftext else "[No text - maybe image/link post]"

    posts_data.append({
        "post_title": post.title,
        "post_url": "https://reddit.com" + post.permalink,
        "comment": None,
        "comment_author": None,
        "comment_score": None,
        "post_body": body,
        "post_score": post.score,
        "post_author": post.author.name if post.author else "[deleted]",
        "created_utc": pd.to_datetime(post.created_utc, unit="s")  # <-- post timestamp
    })

    # Fetch comments
    post.comments.replace_more(limit=0)
    for comment in post.comments.list():
        posts_data.append({
            "post_title": post.title,
            "post_url": "https://reddit.com" + post.permalink,
            "comment": comment.body,
            "comment_author": comment.author.name if comment.author else "[deleted]",
            "comment_score": comment.score,
            "post_body": None,
            "post_score": None,
            "post_author": None,
            "created_utc": pd.to_datetime(comment.created_utc, unit="s")  # <-- comment timestamp
        })

# Save to Google Drive
df = pd.DataFrame(posts_data)

output_path = "/content/drive/MyDrive/reddit_truephilippines1000.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path}")


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Mounted at /content/drive


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Saved 249 rows to /content/drive/MyDrive/reddit_truephilippines1000.csv


# **IV. Exploratory Data Analysis**

Basic EDA to understand the data better.


In [6]:
for f in os.listdir("/content/drive/MyDrive"):
    if "reddit" in f:
        print(f)

reddit_truephilippinesAAA.csv
reddit_truephilippines1000.csv


In [7]:
# Path to the file you saved earlier
file_path = "/content/drive/MyDrive/reddit_truephilippines1000.csv"

# Load CSV
reddit_raw = pd.read_csv(file_path)

In [8]:
print("Shape:", reddit_raw.shape)   # rows x columns
print("\nColumns:", reddit_raw.columns.tolist())
print("\nPreview:")
reddit_raw.head()

Shape: (249, 9)

Columns: ['post_title', 'post_url', 'comment', 'comment_author', 'comment_score', 'post_body', 'post_score', 'post_author', 'created_utc']

Preview:


,post_title,post_url,comment,comment_author,comment_score,post_body,post_score,post_author,created_utc
0,Tax court stops BIR from tapping Cojuangco’s S...,https://reddit.com/r/truePhilippines/comments/...,NaN,NaN,NaN,[No text - maybe image/link post],1.0,intergalacticninja,2025-08-24 13:18:29
1,Ang panget ng Pilipinas,https://reddit.com/r/truePhilippines/comments/...,NaN,NaN,NaN,Gusto ko lang sabihin na sobrang kawawa ng ban...,3.0,Flaky_Weird9588,2025-08-02 07:58:38
2,Ang panget ng Pilipinas,https://reddit.com/r/truePhilippines/comments/...,Agree po tpos po ang taas pa ng tax dito na d ...,BasicCondition1257,3.0,NaN,NaN,NaN,2025-08-06 04:19:17
3,Ang panget ng Pilipinas,https://reddit.com/r/truePhilippines/comments/...,Agree. Marerealized mo lang din naman yan pag ...,CardImpressive2408,1.0,NaN,NaN,NaN,2025-08-07 08:04:51
4,Ang panget ng Pilipinas,https://reddit.com/r/truePhilippines/comments/...,sinabi mo pa hahah may presidente pang paligin...,Any_Cod_9212,1.0,NaN,NaN,NaN,2025-08-16 03:39:04


In [9]:
reddit_raw.tail()

,post_title,post_url,comment,comment_author,comment_score,post_body,post_score,post_author,created_utc
244,ASEAN 'Free Trade Agreeement' 2015 - Packaging...,https://reddit.com/r/truePhilippines/comments/...,NaN,NaN,NaN,[No text - maybe image/link post],0.0,[deleted],2014-11-27 12:04:33
245,Farmers Copying Guns Make Philippines Deadlier...,https://reddit.com/r/truePhilippines/comments/...,NaN,NaN,NaN,[No text - maybe image/link post],5.0,intergalacticninja,2014-11-14 13:45:12
246,Farmers Copying Guns Make Philippines Deadlier...,https://reddit.com/r/truePhilippines/comments/...,"Lito, though, has a valued skill in the event ...",jdb888,1.0,NaN,NaN,NaN,2014-11-15 11:12:21
247,The 5 Most Common Filipino Misconceptions Abou...,https://reddit.com/r/truePhilippines/comments/...,NaN,NaN,NaN,[No text - maybe image/link post],2.0,theigorotjournal,2014-11-07 07:41:42
248,The 5 Most Common Filipino Misconceptions Abou...,https://reddit.com/r/truePhilippines/comments/...,I was just thinking about atheism in the Phili...,[deleted],1.0,NaN,NaN,NaN,2014-11-30 02:06:40


# **V. Social Computing Methods**

**[ WORK IN PROGRESS ]**

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi scelerisque nulla id nulla dapibus, ac suscipit neque convallis. Nullam id suscipit purus. Aenean suscipit dolor in hendrerit varius. Mauris lacinia tincidunt condimentum. Mauris vel ligula quis odio imperdiet iaculis vitae a velit.

# **VI. Results and Conclusion**

**[ WORK IN PROGRESS ]**

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi scelerisque nulla id nulla dapibus, ac suscipit neque convallis. Nullam id suscipit purus. Aenean suscipit dolor in hendrerit varius. Mauris lacinia tincidunt condimentum. Mauris vel ligula quis odio imperdiet iaculis vitae a velit.

# **VII. References**

**[ WORK IN PROGRESS ]**

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi scelerisque nulla id nulla dapibus, ac suscipit neque convallis. Nullam id suscipit purus. Aenean suscipit dolor in hendrerit varius. Mauris lacinia tincidunt condimentum. Mauris vel ligula quis odio imperdiet iaculis vitae a velit.